# 10

In [15]:
import numpy as np
from scipy import stats, optimize

In [ ]:
m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
data = []
for i in range(len(m)):
    data.extend([i] * m[i])


data = np.array(data)
n = len(data)

A = np.arange(0, 10)

In [64]:
def raw2freq (interval, raw_data):
    s = []
    for i in range (len(interval) - 1):
        mask = (interval[i] <= raw_data)  & (raw_data < interval[i + 1])
        s.append(np.sum(mask))
    s.append(np.sum(raw_data == interval[-1]))
    return np.array(s)

## a) 

In [12]:
delta, p_value = stats.kstest(data, 'uniform', args=(0, 10))
print (delta, p_value)

0.14 0.03582511969274871


## b)

In [ ]:
def grouped_normal_mle(x, n):
    x = np.asarray(x)
    n = np.asarray(n)

    a = x - 0.5
    b = x + 0.5

    def neg_log_likelihood(params):
        mu, sigma = params

        if sigma <= 0:
            return np.inf

        p = stats.norm.cdf(b, loc=mu, scale=sigma) - stats.norm.cdf(a, loc=mu, scale=sigma)

        p = np.clip(p, 1e-15, 1.0)

        return -np.sum(n * np.log(p))

    mu0 = np.sum(x * n) / np.sum(n)
    sigma0 = np.sqrt(np.sum(n * (x - mu0)**2) / np.sum(n))

    res = optimize.minimize(
        neg_log_likelihood,
        x0=[mu0, sigma0],
        method='L-BFGS-B',
        bounds=[(None, None), (1e-6, None)]
    )

    return res.x 

In [29]:
print (grouped_normal_mle(A, m))

[4.76999434 2.48865288]


In [ ]:
mu = 4.76999434
sigma = 2.48865288

a = A - 0.5
b = A + 0.5

p = stats.norm.cdf(b, mu, sigma) - stats.norm.cdf(a, mu, sigma)

delta = np.sum((m - n*p)**2 / (n * p))

print(f'delta = {delta}')

delta = 15.712077626988929


In [72]:
delta_sample = []
mu_estimation = 4.76999434
sigma_estimation = 2.48865288
N = 1000000

intervals = np.arange(0, 10, 0.5)

delta_wave = stats.kstest(data, 'norm', (mu_estimation, sigma_estimation)).statistic

for i in range (N):
    samp = np.random.normal(mu_estimation, sigma_estimation, n)

    mu_wave = np.mean(samp)
    sigma_wave = np.std(samp, ddof=1)

    delta_wave_i = stats.kstest(samp, 'norm', (mu_wave, sigma_wave)).statistic
    delta_sample.append(delta_wave_i)

delta_sample = np.array(delta_sample)

print (f'p-value = {np.sum(delta_sample >= delta_wave) / N}')

p-value = 0.011047
